In [ ]:
import os

import Enrich_Job_Offers_Dataframes as de

# ---------------------------------------------------------------------------
# Configuracion externalizada (widgets de Databricks / variables de entorno).
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "job_offers")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")


def _cfg(name: str, default: str = "") -> str:
    """Resuelve un parametro: widget -> variable de entorno -> default."""
    value = dbutils.widgets.get(name)
    return value if value else os.environ.get(name.upper(), default)


CATALOG = _cfg("catalog", "job_offers")
BRONZE_SCHEMA = _cfg("bronze_schema", "bronze")
SILVER_SCHEMA = _cfg("silver_schema", "silver")


def bronze_table(name: str) -> str:
    return f"{CATALOG}.{BRONZE_SCHEMA}.{name}"


def silver_table(name: str) -> str:
    return f"{CATALOG}.{SILVER_SCHEMA}.{name}"


In [ ]:
df=spark.read.table(bronze_table("linkedin"))

In [ ]:
display(df)

## Job Offers

In [ ]:
import pyspark.sql.types as T

offers_df_structure_Schema=T.StructType([
T.StructField("job_id", T.StringType(), True),
T.StructField("job_url", T.StringType(), True),
T.StructField("title", T.StringType(), True),
T.StructField("company_name", T.StringType(), True),
T.StructField("location_city", T.StringType(), True),
T.StructField("location_region", T.StringType(), True),
T.StructField("location_country", T.StringType(), True),
T.StructField("posted_date", T.DateType(), True),
T.StructField("work_mode", T.StringType(), True),
T.StructField("is_salary_available", T.BooleanType(), True),
T.StructField("salary_currency", T.StringType(), True),
T.StructField("salary_min", T.DoubleType(), True),
T.StructField("salary_max", T.DoubleType(), True),
T.StructField("salary_period", T.StringType(), True),
T.StructField("salary_raw", T.StringType(), True),
T.StructField("experience_level", T.StringType(), True),
T.StructField("description_clean", T.StringType(), True),
T.StructField("skills", T.StringType(), True),
T.StructField("source_scraper", T.StringType(), True),
T.StructField("search_role", T.StringType(), True),
T.StructField("scraped_at", T.TimestampType(), True),
T.StructField("_ingest_date", T.DateType(), True)])

In [ ]:
df=df.withColumnsRenamed({
"description_full":"description_clean"
})

In [ ]:
from pyspark.sql import functions as F

n = (
    F.regexp_extract(
        F.lower(F.col("posted_relative")),
        r"hace\s+(\d+)",
        1
    ).cast("int")
)

u = (
    F.when(
        F.col("posted_relative").rlike("(?i)minuto"),
        F.lit("minute")
    )
    .when(
        F.col("posted_relative").rlike("(?i)hora"),
        F.lit("hour")
    )
    .when(
        F.col("posted_relative").rlike("(?i)d[ií]a"),
        F.lit("day")
    )
    .when(
        F.col("posted_relative").rlike("(?i)semana"),
        F.lit("week")
    )
    .when(
        F.col("posted_relative").rlike("(?i)mes"),
        F.lit("month")
    )
    .when(
        F.col("posted_relative").rlike("(?i)a[ñn]o"),
        F.lit("year")
    )
)

df = df.withColumn(
    "posted_date",

    # Si posted_datetime NO está vacío → usarlo
    F.when(
        F.col("posted_datetime").isNotNull() &
        (F.trim(F.col("posted_datetime")) != ""),

        F.to_date(F.col("posted_datetime"))
    )

    # Si posted_datetime está vacío → calcular desde posted_relative
    .otherwise(

        F.when(
            u == "minute",
            F.to_date(
                F.to_timestamp(F.col("scraped_at"))
                - F.make_interval(mins=n)
            )
        )
        .when(
            u == "hour",
            F.to_date(
                F.to_timestamp(F.col("scraped_at"))
                - F.make_interval(hours=n)
            )
        )
        .when(
            u == "day",
            F.date_add(
                F.to_date(F.col("scraped_at")),
                -n
            )
        )
        .when(
            u == "week",
            F.date_add(
                F.to_date(F.col("scraped_at")),
                -n * 7
            )
        )
        .when(
            u == "month",
            F.add_months(
                F.to_date(F.col("scraped_at")),
                -n
            )
        )
        .when(
            u == "year",
            F.add_months(
                F.to_date(F.col("scraped_at")),
                -n * 12
            )
        )
    )
)

In [ ]:
from pyspark.sql.functions import when,col
from pyspark.sql.functions import lit
df=df.withColumns({"is_salary_available":when(col('salary_raw') != "", True).otherwise(False),
                  "source_scraper":lit("Linkedin")})

In [ ]:
from pyspark.sql.functions import col

df_final = df.select(
    [col(c).cast(offers_df_structure_Schema[c].dataType).alias(c) for c in offers_df_structure_Schema.fieldNames()]
)

In [ ]:
df_final.describe()

Función enrich pobla las columnas skills y experience_level a través de la columna 'description_clean' (si ya existe un valor no lo sobreescribe).
La columna employment_type trata de poblarla mediante el mismo campo también.
Para agrupar mejor los distintos roles que se ofertan se genera la columna role_category a través de una lista de keywords.
Por último, se normaliza el salario para poder comparar distintos trabajos con salario anual/mensual/por hora.

In [ ]:
df_completed=de.enrich(df_final)

Normalize Company_Name using 'Title Case' by word

In [ ]:
from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

def normalize_company_name(col: Column) -> Column:
    """Title case por palabra + trim de espacios sobrantes."""
    return F.initcap(F.trim(F.regexp_replace(col, r"\s+", " ")))

def apply_company_normalization(df: DataFrame, col_name: str = "company_name") -> DataFrame:
    """Devuelve un nuevo DataFrame con la columna indicada normalizada."""
    return df.withColumn(col_name, normalize_company_name(F.col(col_name)))

df_completed=apply_company_normalization(df_completed)
# --- Agrupacion de las empresas mas grandes (normalizacion canonica) ---
# Las variantes de una misma compania ("Google" / "Google Ireland Ltd" /
# "Google DeepMind") se agrupan bajo un UNICO nombre para que el conteo de
# ofertas por empresa agregue correctamente en BI.
import re
from pyspark.sql import functions as F

COMPANY_ALIAS_PATTERNS = [
    # --- Big tech ---
    ("Google", r"(?i)\bgoogle\b"),
    ("Amazon", r"(?i)\bamazon\b|\baws\b"),
    ("Microsoft", r"(?i)\bmicrosoft\b"),
    ("Apple", r"(?i)\bapple\b"),
    ("Meta", r"(?i)\bmeta\b"),
    ("IBM", r"(?i)\bibm\b"),
    ("Salesforce", r"(?i)\bsalesforce\b"),
    ("Oracle", r"(?i)\boracle\b"),
    ("Nvidia", r"(?i)\bnvidia\b"),
    ("Intel", r"(?i)\bintel\b"),
    ("Samsung", r"(?i)\bsamsung\b"),
    ("SAP", r"(?i)\bsap\b"),
    ("Logitech", r"(?i)\blogitech\b"),
    ("Nokia", r"(?i)\bnokia\b"),
    ("Lenovo", r"(?i)\blenovo\b"),
    # --- Consultoras y servicios TI ---
    ("Accenture", r"(?i)\baccenture\b"),
    ("Deloitte", r"(?i)\bdeloitte\b"),
    ("Capgemini", r"(?i)\bcapgemini\b"),
    ("Cognizant", r"(?i)\bcognizant\b"),
    ("EPAM", r"(?i)\bepam\b"),
    ("Infosys", r"(?i)\binfosys\b"),
    ("Globant", r"(?i)\bglobant\b"),
    ("McKinsey", r"(?i)\bmckinsey\b|\bquantumblack\b"),
    ("EY", r"(?i)\bey\b|\bernst\s*&\s*young\b"),
    ("PwC", r"(?i)\bpwc\b"),
    ("KPMG", r"(?i)\bkpmg\b"),
    ("UST", r"(?i)\bust\b"),
    ("NTT Data", r"(?i)\bntt\s+data\b"),
    ("Sopra Steria", r"(?i)\bsopra\b"),
    ("Inetum", r"(?i)\binetum\b"),
    ("Devoteam", r"(?i)\bdevoteam\b"),
    ("Adecco", r"(?i)\badecco\b"),
    ("CGI", r"(?i)\bcgi\b"),
    ("Logicalis", r"(?i)\blogicalis\b"),
    ("Aubay", r"(?i)\baubay\b"),
    ("Indra", r"(?i)\bindra\b"),
    ("Izertis", r"(?i)\bizertis\b"),
    ("Babel", r"(?i)\bbabel\b"),
    ("Manpower", r"(?i)\bmanpower(?:group)?\b"),
    ("Randstad", r"(?i)\brandstad\b"),
    ("Gi Group", r"(?i)\bgi\s+group\b"),
    ("Hays", r"(?i)\bhays\b"),
    ("Teleperformance", r"(?i)\bteleperformance\b"),
    ("Acciona", r"(?i)\bacciona\b"),
    ("Criteo", r"(?i)\bcriteo\b"),
    ("Adevinta", r"(?i)\badevinta\b"),
    # --- Banca, seguros y servicios financieros ---
    ("JPMorgan Chase", r"(?i)\bjp\s*morgan(?:chase)?\b"),
    ("Morgan Stanley", r"(?i)\bmorgan\s+stanley\b"),
    ("Mastercard", r"(?i)\bmastercard\b"),
    ("Santander", r"(?i)\bsantander\b"),
    ("BBVA", r"(?i)\bbbva\b"),
    ("CaixaBank", r"(?i)\bcaixabank\b|\bvidacaixa\b"),
    ("ABN AMRO", r"(?i)\babn\s+amro\b"),
    ("Rabobank", r"(?i)\brabobank\b"),
    ("Deutsche Bank", r"(?i)\bdeutsche\s+bank\b"),
    ("Deutsche Telekom", r"(?i)\bdeutsche\s+telekom\b"),
    ("Zurich", r"(?i)\bzurich\b"),
    ("Northern Trust", r"(?i)\bnorthern\s+trust\b"),
    ("Marsh", r"(?i)\bmarsh\b"),
    ("UPMC", r"(?i)\bupmc\b"),
    ("TD SYNNEX", r"(?i)\btd\s+synnex\b"),
    # --- Telecom, transporte y energia ---
    ("Vodafone", r"(?i)\bvodafone(?:ziggo)?\b"),
    ("Telefonica", r"(?i)\btelef[oó]nica\b"),
    ("Orange", r"(?i)\borange\b(?!\s+quarter)"),
    ("T-Mobile", r"(?i)\bt[- ]?mobile\b"),
    ("Verizon", r"(?i)\bverizon\b"),
    ("Boeing", r"(?i)\bboeing\b"),
    ("Airbus", r"(?i)\bairbus\b"),
    # --- Industrial y hardware ---
    ("Siemens", r"(?i)\bsiemens\b"),
    ("Bosch", r"(?i)\bbosch\b"),
    ("ABB", r"(?i)\babb\b"),
    ("Abbott", r"(?i)\babbott\b"),
    ("Philips", r"(?i)\bphilips\b"),
    ("Lockheed Martin", r"(?i)\blockheed\b"),
    ("Northrop Grumman", r"(?i)\bnorthrop\b"),
    ("Honeywell", r"(?i)\bhoneywell\b"),
    ("Vertiv", r"(?i)\bvertiv\b"),
    ("Stryker", r"(?i)\bstryker\b"),
    ("Analog Devices", r"(?i)\banalog\s+devices\b"),
    ("Bentley Systems", r"(?i)\bbentley\s+systems\b"),
    ("Keysight", r"(?i)\bkeysight\b"),
    ("Keyrus", r"(?i)\bkeyrus\b"),
    ("Keyrock", r"(?i)\bkeyrock\b"),
    ("Disney", r"(?i)\bdisney\b"),
    # --- Farma y consumo ---
    ("Roche", r"(?i)\broche\b"),
    ("Thermo Fisher", r"(?i)\bthermo\s+fisher\b"),
    ("Novartis", r"(?i)\bnovartis\b"),
    ("AstraZeneca", r"(?i)\bastrazeneca\b"),
    ("Pfizer", r"(?i)\bpfizer\b"),
    ("Sanofi", r"(?i)\bsanofi\b"),
    ("Merck", r"(?i)\bmerck\b"),
    ("Bayer", r"(?i)\bbayer\b"),
    ("Johnson & Johnson", r"(?i)\bjohnson\s*&\s*johnson\b|\bjanssen\b"),
    ("Bristol Myers Squibb", r"(?i)\bbristol\b"),
    ("Eli Lilly", r"(?i)\blilly\b"),
    ("Takeda", r"(?i)\btakeda\b"),
    ("Medtronic", r"(?i)\bmedtronic\b"),
    ("PepsiCo", r"(?i)\bpepsico\b"),
    ("Procter & Gamble", r"(?i)\bprocter\b"),
    ("Nestle", r"(?i)\bnestl[ée]\b"),
    ("Unilever", r"(?i)\bunilever\b"),
    ("Danone", r"(?i)\bdanone\b"),
    ("Coca-Cola", r"(?i)\bcoca\s*-?\s*cola\b"),
    ("Heineken", r"(?i)\bheineken\b"),
    ("Hershey", r"(?i)\bhershey\b"),
    ("Philip Morris", r"(?i)\bphilip\s+morris\b"),
    ("Nike", r"(?i)\bnike\b"),
    ("Adidas", r"(?i)\badidas\b"),
    # --- Internet, SaaS y retail ---
    ("Fever", r"(?i)\bfever(?:up)?\b"),
    ("Glovo", r"(?i)\bglovo\b"),
    ("Revolut", r"(?i)\brevolut\b(?!\s+f1)"),
    ("Booking", r"(?i)\bbooking\b"),
    ("Expedia", r"(?i)\bexpedia\b"),
    ("Adyen", r"(?i)\badyen\b"),
    ("Stripe", r"(?i)\bstripe\b"),
    ("Twilio", r"(?i)\btwilio\b"),
    ("Databricks", r"(?i)\bdatabricks\b"),
    ("Snowflake", r"(?i)\bsnowflake\b"),
    ("ServiceNow", r"(?i)\bservicenow\b"),
    ("Workday", r"(?i)\bworkday\b"),
    ("GitHub", r"(?i)\bgithub\b"),
    ("CrowdStrike", r"(?i)\bcrowdstrike\b"),
    ("Uber", r"(?i)\buber\b"),
    ("Carrefour", r"(?i)\bcarrefour\b"),
    ("Lidl", r"(?i)\blidl\b"),
    ("IKEA", r"(?i)\bikea\b"),
    ("Just Eat Takeaway", r"(?i)\bjust\s+eat\b"),
    ("Zalando", r"(?i)\bzalando\b"),
]


def company_name_canonical(name: str) -> str:
    """Nombre canonico de empresa (first-match wins)."""
    value = (name or "").strip()
    for canon, pattern in COMPANY_ALIAS_PATTERNS:
        if re.search(pattern, value):
            return canon
    return value

def apply_company_canonical(df, col_name: str = "company_name"):
    """Agrupa las variantes de las grandes companias en un unico nombre
    canonico (Google / Google Ireland Ltd / Google DeepMind -> "Google").

    Identico resultado al de la funcion pura company_name_canonical().

    RENDIMIENTO (importante): la expresion con los ~110 patrones se evalua
    SOLO sobre los valores DISTINTOS de company_name (centenares/miles) y
    luego se hace un join, en vez de aplicar los 110 regex por fila a toda
    la tabla. Asi el coste no depende del numero de filas.
    """
    from pyspark.sql import functions as F

    raw = F.col(col_name)
    base = F.trim(F.coalesce(raw, F.lit("")))
    expr = base
    for canon, pattern in COMPANY_ALIAS_PATTERNS:
        expr = F.when(base.rlike(pattern), F.lit(canon)).otherwise(expr)
    # Los NULL se conservan tal cual (el join por clave null no matchea y
    # el coalesce devuelve el valor original).
    expr = F.when(raw.isNull(), raw).otherwise(expr)

    mapped = (df.select(col_name)
                .distinct()
                .withColumn("__company_canon", expr))
    return (df.join(mapped, col_name, "left")
              .withColumn(col_name,
                          F.coalesce(F.col("__company_canon"), F.col(col_name)))
              .drop("__company_canon"))
df_completed=apply_company_canonical(df_completed)

Function to drop duplicates by Job_Title, Company_Name and Location_City

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

def add_semantic_key(df):
    return df.withColumn(
        "_key",
        F.lower(F.trim(F.regexp_replace(
            F.concat_ws("|", F.col("title"), F.col("company_name"), F.col("location_city")),
            r"\s+", " "))),
    )

w = Window.partitionBy("_key").orderBy(F.desc("posted_date"), F.desc("_ingest_date"))

df_completed = (add_semantic_key(df_completed)
           .withColumn("_rn", F.row_number().over(w))
           .filter(F.col("_rn") == 1)
           .drop("_key", "_rn"))

In [ ]:
df_completed=df_completed.drop(*['scraped_at','_ingest_date','salary_period','work_mode'])

In [ ]:
df_completed=df_completed.withColumnsRenamed({"salary_period_norm":"salary_period","work_mode_norm":"work_mode"})

In [ ]:
df_completed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table("offers_linkedin"))

Dim_Companies

In [ ]:
companies_df=df.select(["company_name","company_size","company_industry","company_description"])
companies_df=apply_company_normalization(companies_df)
companies_df=apply_company_canonical(companies_df)

In [ ]:
from pyspark.sql import functions as F

groups = [
    (2, 10, "2-10"),
    (11, 50, "11-50"),
    (51, 200, "51-200"),
    (201, 500, "201-500"),
    (501, 1000, "501-1000"),
    (1001, 5000, "1001-5000"),
    (5001, 10000, "5001-10000"),
]

text = F.lower(F.trim(F.col("company_size")))

# Caso A: el texto ya es un rango -> "1001-5000", "1001 - 5000 empleados", "+10000"
rango = F.when(
    text.rlike(r"^\+\d+"),
    F.regexp_extract(text, r"^(\+\d+)", 1)
).when(
    text.rlike(r"^\d+\s*-\s*\d+"),
    F.regexp_replace(F.regexp_extract(text, r"^(\d+\s*-\s*\d+)", 1), r"\s*-\s*", "-")
).otherwise(None)

# Caso B: número exacto -> "250" en el texto, o el valor de num_empleados
num = F.coalesce(
    F.when(text.rlike(r"^\d+$"), text.try_cast("int")),
    F.col("company_size").try_cast("int")   # <- aquí estaba el error
)

# Bucket del número exacto
expr = F.when(num > 10000, "+10000")
for lo, hi, nombre in groups:
    expr = expr.when(num.between(lo, hi), nombre)
expr = expr.otherwise(None)

# Prioriza el rango ya escrito; si no, calcula por número
companies_df = companies_df.withColumn("company_size_clean", F.when(F.coalesce(rango, expr).isNull(), "unknown").otherwise(F.coalesce(rango, expr)))

In [ ]:
companies_df=companies_df.dropDuplicates(["company_name"]).drop("company_size").withColumnRenamed("company_size_clean","company_size").drop("company_size_clean")

In [ ]:
companies_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table("companies_linkedin"))